# VAE

VAE は、入力を小さな潜在変数 `z` へ圧縮し、その `z` から入力を再構成する生成モデルです。ただし、1 点の `z` へ直接写すのではなく、入力ごとに潜在分布 `q(z|x)` を作り、そこからサンプルした `z` をデコードして `x` を戻します。

損失は 2 つの圧力を持ちます。再構成項は戻した `x_hat` を元の `x` に近づけます。KL 項は潜在分布を標準正規 `N(0,1)` に近づけます。ELBO は、この再構成の良さと潜在空間の整い方を同時に見る目的関数です。再パラメータ化 `z = mu + sigma * eps` により、乱数を使いながら `mu` と `sigma` へ勾配を通せます。この教材では、再構成を良くする力と潜在空間を整える力の釣り合いを読みます。

In [ ]:
import math
import random
import statistics

random.seed(33)


def make_data(n=120):
    xs = []
    for _ in range(n):
        if random.random() < 0.5:
            xs.append(random.gauss(-2.0, 0.45))
        else:
            xs.append(random.gauss(2.2, 0.55))
    return xs

data = make_data()
print('mean:', round(statistics.mean(data), 3))
print('std:', round(statistics.pstdev(data), 3))
print('left ratio:', round(sum(x < 0 for x in data) / len(data), 3))

エンコーダは `mu(x)` と `logvar(x)` を出す。デコーダは `z` から `x_hat` を作る。1 次元の線形モデルでも、VAE の損失構造は確認できる。`mu` は入力ごとの潜在位置、`logvar` はその周りの曖昧さを表し、デコーダはサンプルされた潜在から観測値を戻す。

In [ ]:
def encode(x, params):
    a, b, c, d, _, _ = params
    mu = a * x + b
    logvar = max(-6.0, min(4.0, c * x + d))
    return mu, logvar


def decode(z, params):
    _, _, _, _, m, n = params
    return m * z + n


def kl_standard_normal(mu, logvar):
    return 0.5 * (math.exp(logvar) + mu * mu - 1.0 - logvar)


def reparameterize(mu, logvar, eps):
    return mu + math.exp(0.5 * logvar) * eps

params0 = [0.25, 0.0, -0.15, -1.0, 0.8, 0.0]
for x in [-2.0, 0.0, 2.0]:
    mu, logvar = encode(x, params0)
    print(x, 'mu=', round(mu, 3), 'var=', round(math.exp(logvar), 3), 'KL=', round(kl_standard_normal(mu, logvar), 3))

再構成項は `x_hat` が `x` に近いほど良い。KL は `q(z|x)` が `N(0,1)` から離れるほど大きくなる。beta-VAE では `beta * KL` として KL の強さを変える。再構成だけを下げると潜在は入力ごとに散らばりやすく、KL だけを強くすると入力差を持ちにくくなる。

In [ ]:
def evaluate(data, params, beta=1.0, recon_var=0.35, eps_list=None):
    if eps_list is None:
        eps_list = [0.0 for _ in data]
    recon_terms = []
    kl_terms = []
    for x, eps in zip(data, eps_list):
        mu, logvar = encode(x, params)
        z = reparameterize(mu, logvar, eps)
        x_hat = decode(z, params)
        recon = 0.5 * ((x - x_hat) ** 2 / recon_var + math.log(2 * math.pi * recon_var))
        recon_terms.append(recon)
        kl_terms.append(kl_standard_normal(mu, logvar))
    recon = statistics.mean(recon_terms)
    kl = statistics.mean(kl_terms)
    loss = recon + beta * kl
    return loss, recon, kl

eps_fixed = [random.gauss(0.0, 1.0) for _ in data]
print('initial beta=1:', [round(v, 3) for v in evaluate(data, params0, beta=1.0, eps_list=eps_fixed)])
print('initial beta=4:', [round(v, 3) for v in evaluate(data, params0, beta=4.0, eps_list=eps_fixed)])

`eps` は外から来る標準正規ノイズで、学習対象ではありません。`z = mu + sigma * eps` と分けることで、`mu` や `logvar` を動かしたときの変化を追えます。乱数を直接学習対象にせず、分布の位置と幅へ勾配を通すための書き換えが再パラメータ化です。

In [ ]:
mu = 0.7
logvar = -0.8
eps_values = [-1.0, 0.0, 1.0]
for eps in eps_values:
    z = reparameterize(mu, logvar, eps)
    print('eps=', eps, 'z=', round(z, 3))

有限差分で小さな VAE を学習する。目的は `recon + beta*KL` の最小化で、beta を変えると再構成と潜在整理のバランスが変わる。

In [ ]:
def finite_grad(data, params, beta, eps_list, h=1e-3):
    grads = []
    for i in range(len(params)):
        plus = params[:]
        minus = params[:]
        plus[i] += h
        minus[i] -= h
        lp = evaluate(data, plus, beta=beta, eps_list=eps_list)[0]
        lm = evaluate(data, minus, beta=beta, eps_list=eps_list)[0]
        grads.append((lp - lm) / (2.0 * h))
    return grads


def train_vae(data, beta=1.0, steps=180, lr=0.025):
    params = [0.2, 0.0, -0.08, -1.0, 0.8, 0.0]
    history = []
    eps_list = [random.gauss(0.0, 1.0) for _ in data]
    for step in range(steps):
        grads = finite_grad(data, params, beta, eps_list)
        for i, g in enumerate(grads):
            params[i] -= lr * g
        params[2] = max(-1.5, min(1.5, params[2]))
        params[3] = max(-4.0, min(2.0, params[3]))
        if step % 45 == 0 or step == steps - 1:
            history.append((step, *evaluate(data, params, beta=beta, eps_list=eps_list)))
    return params, history

params_b1, hist_b1 = train_vae(data, beta=1.0)
for row in hist_b1:
    step, loss, recon, kl = row
    print(step, 'loss=', round(loss, 3), 'recon=', round(recon, 3), 'KL=', round(kl, 3))

beta を大きくすると KL が強く抑えられ、潜在は標準正規へ近づく。一方で入力ごとの情報を持ちにくくなり、再構成は悪くなりやすい。

In [ ]:
params_b4, hist_b4 = train_vae(data, beta=4.0)

last_b1 = hist_b1[-1]
last_b4 = hist_b4[-1]
print('beta=1:', tuple(round(v, 3) if isinstance(v, float) else v for v in last_b1))
print('beta=4:', tuple(round(v, 3) if isinstance(v, float) else v for v in last_b4))


def latent_stats(data, params):
    mus = []
    vars_ = []
    for x in data:
        mu, logvar = encode(x, params)
        mus.append(mu)
        vars_.append(math.exp(logvar))
    return {
        'mu_std': statistics.pstdev(mus),
        'var_mean': statistics.mean(vars_),
        'kl': evaluate(data, params, beta=1.0)[2],
    }

print('latent beta=1:', {k: round(v, 3) for k, v in latent_stats(data, params_b1).items()})
print('latent beta=4:', {k: round(v, 3) for k, v in latent_stats(data, params_b4).items()})

学習後は `z ~ N(0,1)` をデコードして生成する。補間では、2 つの入力の潜在平均をつないでデコードし、潜在空間の連続性を見る。標準正規から取った点が自然にデコードできるほど、KL で整えた潜在空間を生成に使いやすくなる。

In [ ]:
def generate(params, n=8):
    out = []
    for _ in range(n):
        z = random.gauss(0.0, 1.0)
        out.append((z, decode(z, params)))
    return out

print('generated beta=1:', [(round(z, 2), round(x, 2)) for z, x in generate(params_b1)])

x_left = min(data)
x_right = max(data)
mu_l, _ = encode(x_left, params_b1)
mu_r, _ = encode(x_right, params_b1)
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    z = (1.0 - alpha) * mu_l + alpha * mu_r
    print('alpha=', alpha, 'z=', round(z, 3), 'decode=', round(decode(z, params_b1), 3))

Posterior collapse は、潜在変数を置いたのに `q(z|x)` が入力差をほとんど持たない状態です。つまり、どの入力を入れても似た潜在分布になり、decoder が `z` をあまり使わなくなります。KL が極端に小さく、`mu` のばらつきも小さい場合は兆候として疑います。

In [ ]:
params_b12, hist_b12 = train_vae(data, beta=12.0)
stats_b12 = latent_stats(data, params_b12)
print('beta=12 final:', tuple(round(v, 3) if isinstance(v, float) else v for v in hist_b12[-1]))
print('latent beta=12:', {k: round(v, 4) for k, v in stats_b12.items()})
if stats_b12['kl'] < 0.05 and stats_b12['mu_std'] < 0.2:
    print('collapse warning: latent carries little input variation')
else:
    print('collapse warning: not triggered')

VAE は、再構成したい圧力と、標準正規から生成しやすい潜在空間へ整える圧力を同時に扱う。beta を上げるほど潜在は整うが、入力情報を落としすぎると collapse に近づく。